# V7: new readers AND unseen sentences
Bounded exploratory diagnostic, not a search. The existing CNN is frozen: raw word-level TRT_total (not continuous EEG), LR=0.0001, seeds 45/62, same 100-epoch maximum and macro-F1 checkpoint rule.
One fixed balanced sentence split: **66 fit / 22 validation / 22 unseen test**. Each job fits ten readers on fit sentences, validates on a separate reader and separate sentences, then tests a held-out reader. Test sentences are excluded from ALL fitting and checkpoint-selection data.
The same model is also scored on the held-out reader's 66 fit-sentence identities as a seen-stimulus comparison, without extra training. Those are new-reader observations, not training examples.
Two target conditions: true sentiment, and one fixed arbitrary sentence mapping, shuffled independently within each split to preserve class balance and shared across readers. The mapping seed is independent of both initialization seeds. **48 fits / 96 evaluation rows**, roughly 1–2 hours on a GPU (estimate; actual runtime varies). Saves after every fit; resumes completed fits.
Limitations: only one sentence split and one random mapping; 22 unseen sentences, not 528 independent stimuli. Settings were selected in earlier work that used these sentences, so this is NOT a pristine external test or a formal permutation significance test. Smaller fit/validation sets mean old 0.670 cannot be compared as if only sentence overlap changed. Seen/unseen views share a trained model but contain different sentences and sample counts. No result by itself proves neural sentiment specificity.

## 1. Setup
Choose **Runtime → Change runtime type → GPU** first. This notebook uses the fixed tag `v7-unseen-sentences-v1`. No dependency installation is performed.

In [ ]:
import os, sys, subprocess, json
import tensorflow as tf
assert tf.config.list_physical_devices('GPU'), 'No GPU: change runtime type to GPU, reconnect, then rerun setup.'
from google.colab import drive
drive.mount('/content/drive')
repo = '/content/teco-v7-unseen-sentences'
run_ref = 'v7-unseen-sentences-v1'
if not os.path.isdir(repo):
    subprocess.run(['git', 'clone', '-q', '--branch', run_ref, '--single-branch',
                    'https://github.com/parmisbathayan/reproduce-teco-results.git', repo], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == subprocess.check_output(['git', 'rev-parse', run_ref + '^{commit}'], text=True).strip()
assert not subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip(), 'Use a clean fixed revision.'
print('GPU ready. Fixed Git commit:', commit, flush=True)
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
out_dir = os.path.join(base, 'v7_unseen_sentences_45_62')
reference_dir = os.path.join(base, 'v4_low_lr_confirmation_45_62')
command = [sys.executable, '-u', '-B', 'src/v7_unseen_sentences.py',
           '--trt-dir', '/content/drive/MyDrive/Thesis/Data/TRT_Total',
           '--labels-csv', '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv',
           '--out-dir', out_dir, '--config', 'configs/v7_unseen_sentences.json',
           '--reference-out-dir', reference_dir]
def stream_run(arguments):
    with subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            print('Stopped. Completed fits are saved; rerun setup, preflight and training to resume.', flush=True)
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, arguments)


## 2. Preflight — no training
Checks data/code/runtime against the completed CNN; creates one fixed balanced split and mapping; rejects duplicate normalized sentence texts and exact nonzero-array overlap across fitting, validation, and held-out-reader evaluations. All-zero cases are recorded, not silently removed. These checks do not exclude near-duplicates or upstream preprocessing contamination. Stop and report any failure; do not bypass it.

In [ ]:
stream_run(command + ['--prepare-only'])
import pandas as pd
from IPython.display import display
plan = pd.read_csv(os.path.join(out_dir, 'sentence_plan.csv'))
display(plan.groupby('partition')[['sentiment', 'shuffled_sentence_labels']].agg(['count', 'sum']))
print('sum = number of positive labels; each is half the count. Preflight finished; no training yet.')


## 3. Run or resume the 48 fits
Each fit prints START, a heartbeat every 30 seconds while training, and SAVED with seen/unseen scores. Both evaluations are saved together. After a disconnect, rerun sections 1–3; completed fits are skipped. Do not change seeds, split, or parameters in response to results.

In [ ]:
stream_run(command)


## 4. Verify and display results
Reports both accuracy/correct counts and mean per-reader macro-F1. Correct-count differences between real/random targets are descriptive and use different target assignments. Predictions across readers/seeds share stimuli. No p-value is computed. Send the results for review; do not start V6 or more variants automatically.

In [ ]:
from pathlib import Path
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
sys.path.insert(0, os.path.join(repo, 'src'))
from v7_unseen_sentences import completed_jobs, write_summaries
from data import PARTICIPANTS
config = json.loads(Path('configs/v7_unseen_sentences.json').read_text())
codes = [code for _, code in PARTICIPANTS]
metrics = pd.read_csv(os.path.join(out_dir, 'fold_metrics.csv'))
assert len(metrics) == 96 and len(completed_jobs(metrics, codes, config)) == 48
plan = pd.read_csv(os.path.join(out_dir, 'sentence_plan.csv'))
assert plan.sentence_id.nunique() == 110
for row in metrics.itertuples(index=False):
    split = 'fit' if row.evaluation == 'seen_sentences' else 'unseen_test'
    expected = plan[plan.partition == split].set_index('sentence_id')[row.condition]
    filename = f'{row.condition}_seed_{row.seed}_{row.participant}_{row.evaluation}.csv'
    prediction = pd.read_csv(os.path.join(out_dir, 'predictions', filename))
    assert len(prediction) == row.n_test == len(expected)
    assert prediction.sentence_id.nunique() == len(expected)
    assert set(prediction.sentence_id) == set(expected.index)
    assert np.array_equal(prediction.true_label, expected.loc[prediction.sentence_id].to_numpy())
    assert np.array_equal(prediction.predicted_label, prediction[['probability_negative','probability_positive']].to_numpy().argmax(axis=1))
    assert abs(f1_score(prediction.true_label, prediction.predicted_label, average='macro') - row.macro_f1) < 1e-10
    assert abs(accuracy_score(prediction.true_label, prediction.predicted_label) - row.accuracy) < 1e-10
    assert int((prediction.true_label == prediction.predicted_label).sum()) == row.correct
    assert row.n_test_negative == row.n_test_positive == row.n_test // 2
write_summaries(metrics, out_dir, codes, config)
display(pd.read_csv(os.path.join(out_dir, 'summary_overall.csv')))
display(pd.read_csv(os.path.join(out_dir, 'summary_by_seed.csv')))
seen = metrics[metrics.evaluation == 'seen_sentences']
unseen = metrics[metrics.evaluation == 'unseen_sentences']
paired = seen.merge(unseen, on=['condition','seed','participant'], suffixes=('_seen','_unseen'), validate='one_to_one')
paired.to_csv(os.path.join(out_dir, 'paired_seen_vs_unseen.csv'), index=False)
real = metrics[metrics.condition == 'sentiment']
shuffled = metrics[metrics.condition == 'shuffled_sentence_labels']
contrast = real.merge(shuffled, on=['seed','participant','evaluation'], suffixes=('_real','_shuffled'), validate='one_to_one')
assert len(contrast) == 48
for field in ['validation_participant','fit_participants','n_fit','n_validation','n_test','sentence_plan_sha256']:
    assert (contrast[field + '_real'] == contrast[field + '_shuffled']).all()
contrast.to_csv(os.path.join(out_dir, 'paired_real_vs_shuffled.csv'), index=False)
print('Verified all 96 evaluation files. Complete: 48 fits; review these results before any further run.')
